In [1]:
# ============================================================
# Cell 1: 环境准备
# ============================================================
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
driver = get_driver()
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# ============================================================
# Cell 2: 确保 Stage2 demo 数据（幂等）
# ============================================================
from src.engineering_intelligence.strategy_classifier import create_engineering_strategy
from src.engineering_intelligence.construct_service import create_engineered_construct
from src.engineering_intelligence.claim_extractor import create_technical_claim
from src.engineering_intelligence.result_extractor import create_technical_result
from src.engineering_intelligence.technology_assessment import create_technology_assessment
from src.shared.evidence_source import get_or_create_pathogen

STRATEGY_DEFS = [
    ("host_range_expansion",   "通过改造尾纤维或受体结合蛋白扩展宿主范围"),
    ("tail_fiber_engineering", "尾纤维定向改造，改变受体识别特异性"),
    ("lysis_enhancement",      "改造裂解模块或裂解酶，增强裂解活性"),
    ("biofilm_disruption",     "引入降解胞外基质的酶或改造穿透能力"),
    ("payload_delivery",       "递送抗耐药 / 抗毒力 / CRISPR 等外源载荷"),
]
strategy_ids = {}
for stype, desc in STRATEGY_DEFS:
    strategy_ids[stype] = create_engineering_strategy(
        driver, strategy_type=stype, description=desc,
        evidence_maturity="in_vitro", actor_id="demo",
    )
print(f"✅ 策略：{len(strategy_ids)} 个")

for sp in ["Klebsiella pneumoniae", "Pseudomonas aeruginosa", "Staphylococcus aureus", "Escherichia coli"]:
    get_or_create_pathogen(driver, species=sp, pathogen_type="bacteria", actor_id="demo")
print("✅ 病原体就绪")

CONSTRUCT_DEFS = [
    ("host_range_expansion",   "vB_Kpn_HRE_001", "Klebsiella pneumoniae", "in_vitro_tested", "2024-06-10"),
    ("host_range_expansion",   "vB_Pae_HRE_002", "Pseudomonas aeruginosa","in_vitro_tested", "2024-09-01"),
    ("tail_fiber_engineering", "vB_Kpn_TFE_003", "Klebsiella pneumoniae", "in_vitro_tested", "2024-06-10"),
    ("tail_fiber_engineering", "vB_Eco_TFE_004", "Escherichia coli",      "proposed",        "2025-01-10"),
    ("lysis_enhancement",      "vB_Pae_LYE_005", "Pseudomonas aeruginosa","in_vitro_tested", "2024-09-01"),
    ("lysis_enhancement",      "vB_Sau_LYE_006", "Staphylococcus aureus", "in_vitro_tested", "2024-09-01"),
    ("biofilm_disruption",     "vB_Pae_BFD_007", "Pseudomonas aeruginosa","in_vitro_tested", "2024-09-01"),
    ("biofilm_disruption",     "vB_Sau_BFD_008", "Staphylococcus aureus", "proposed",        "2024-09-01"),
    ("payload_delivery",       "vB_Kpn_PLD_009", "Klebsiella pneumoniae", "in_vitro_tested", "2024-05-20"),
    ("payload_delivery",       "vB_Pae_PLD_010", "Pseudomonas aeruginosa","proposed",        "2024-05-20"),
]

def _get_construct_id(name):
    with driver.session() as s:
        r = s.run("MATCH (ec:EngineeredPhageConstruct {public_name: $n}) RETURN ec.construct_id AS id LIMIT 1", n=name).single()
        return r["id"] if r else None

construct_ids = {}
for stype, name, target_sp, status, pub_date in CONSTRUCT_DEFS:
    existing = _get_construct_id(name)
    if existing:
        construct_ids[name] = (existing, stype)
        continue
    cid = create_engineered_construct(driver, public_name=name, construct_code=name.upper(),
        parent_phage_name=None, intended_effects=[stype], target_pathogen_species=[target_sp],
        strategy_ids=[strategy_ids[stype]], construct_status=status, first_public_date=pub_date, actor_id="demo")
    construct_ids[name] = (cid, stype)
print(f"✅ 构建体：{len(construct_ids)} 个")

CLAIM_TYPE_MAP = {"host_range_expansion": "host_range", "tail_fiber_engineering": "host_range", "lysis_enhancement": "efficacy", "biofilm_disruption": "efficacy", "payload_delivery": "safety"}
for name, (cid, stype) in construct_ids.items():
    with driver.session() as s:
        existing = s.run("MATCH (tr:TechnicalResult)-[:RESULT_FOR]->(ec:EngineeredPhageConstruct {construct_id: $cid}) RETURN count(tr) AS c", cid=cid).single()["c"]
    if existing: continue
    ct = CLAIM_TYPE_MAP[stype]
    create_technical_claim(driver, claim_type=ct, claim_text=f"{name} 在体外展示出改进的 {ct} 表型",
        claimant_type="publication", evidence_context="in_vitro", construct_id=cid, strategy_id=strategy_ids[stype], actor_id="demo")
    create_technical_result(driver, result_type=ct, study_context="in_vitro", outcome_direction="positive",
        metric_name="relative_improvement", metric_value=2.0, metric_unit="fold", comparator="亲本噬菌体",
        sample_size=10, limitation_summary="仅体外数据", reproducibility_status="single_source", construct_id=cid, actor_id="demo")
    create_technology_assessment(driver, subject_type="construct", subject_id=cid,
        evidence_maturity="in_vitro", technical_relevance="high", translational_potential="medium",
        manufacturability_risk="medium", safety_uncertainty="low", ip_relevance="medium",
        internal_capability_gap="high" if stype in ("payload_delivery", "biofilm_disruption") else "medium",
        assessment_summary=f"{name} 的体外证据支持其技术相关性。", actor_id="demo")
print("✅ 主张 / 结果 / 评估 就绪")

ℹ️ 策略类型 'host_range_expansion' 已存在，ID: ENG:STRAT:9361AFF5
ℹ️ 策略类型 'tail_fiber_engineering' 已存在，ID: ENG:STRAT:723A98E1
ℹ️ 策略类型 'lysis_enhancement' 已存在，ID: ENG:STRAT:F11EF640
ℹ️ 策略类型 'biofilm_disruption' 已存在，ID: ENG:STRAT:35E50233
ℹ️ 策略类型 'payload_delivery' 已存在，ID: ENG:STRAT:3E45912A
✅ 策略：5 个
✅ 病原体就绪
✅ 构建体：10 个
✅ 主张 / 结果 / 评估 就绪


In [3]:
# ============================================================
# Cell 3: 为策略生成技术简报
# ============================================================
from src.engineering_intelligence.strategy_classifier import get_all_strategies
from src.engineering_intelligence.technology_brief import generate_technology_brief

strategies = get_all_strategies(driver)
print(f"当前策略数：{len(strategies)}\n")
brief_ids = []
for s in strategies:
    stype = s["strategy_type"]
    brief = generate_technology_brief(driver, strategy_id=s["id"],
        technology_question=f"{stype} 的公开证据成熟度如何？", persist=True)
    brief_ids.append(brief["brief_id"])
    print(f"📄 {stype}: {brief['brief_id']}, 构建体 {len(brief['known_constructs'])}")
print(f"\n✅ 共生成 {len(brief_ids)} 份技术简报")

当前策略数：5

📄 biofilm_disruption: CI:BRIEF:2AF6C39A, 构建体 2
📄 host_range_expansion: CI:BRIEF:0D1A5F74, 构建体 2
📄 lysis_enhancement: CI:BRIEF:E0B24727, 构建体 2
📄 payload_delivery: CI:BRIEF:D3F2D45A, 构建体 2
📄 tail_fiber_engineering: CI:BRIEF:9CE8150F, 构建体 2

✅ 共生成 5 份技术简报


In [4]:
# ============================================================
# Cell 4: DeepSeek 技术候选抽取
# ============================================================
from src.shared.evidence_source import create_source_artifact
from src.engineering_intelligence.claim_extractor import extract_technical_candidates, persist_candidates

src_id = create_source_artifact(driver, source_type="literature",
    title="Engineered phage with expanded host range against KL47 K. pneumoniae",
    url="https://example.com/paper-kl47-engineered-phage",
    published_date="2025-08-01", credibility_tier="secondary", actor_id="demo")
print(f"✅ SourceArtifact: {src_id}")

SAMPLE_TEXT = """
We report an engineered bacteriophage vB_Kpn_HRE_001 derived from the wild-type phage PKP001,
in which the tail fiber gene was replaced with a chimeric tail fiber to expand host range.
The engineered construct was tested in vitro against 12 clinical isolates of KL47-type K. pneumoniae.
The engineered phage demonstrated a 2.0-fold improvement in host coverage compared with the parental phage,
with a lysis efficiency of 98.5% in the tested strains. No cytotoxicity was observed in human cell lines.
However, the study is limited to in vitro assays, and in vivo efficacy remains to be established.
"""

extraction = extract_technical_candidates(driver, source_artifact_id=src_id, source_text=SAMPLE_TEXT, actor_id="demo")
print("📊 抽取统计：", json.dumps(extraction["stats"], ensure_ascii=False))

ids = persist_candidates(driver, extraction, actor_id="demo")
print("📦 落库结果：")
for k, v in ids.items():
    print(f"  {k}: {len(v)} 条")

✅ SourceArtifact: SRC:9D286015
📊 抽取统计： {"strategies": 1, "constructs": 1, "claims": 1, "results": 0, "repaired": false, "parse_error": false}
📦 落库结果：
  strategy_ids: 1 条
  construct_ids: 0 条
  claim_ids: 0 条
  result_ids: 0 条


In [5]:
# ============================================================
# Cell 5: 列出待审候选 + 审核通过/拒绝
# ============================================================
from src.engineering_intelligence.claim_extractor import list_pending_candidates, approve_candidate, reject_candidate

pending = list_pending_candidates(driver)
print("待审：", {k: len(v) for k, v in pending.items()})

if pending["claims"]:
    cid = pending["claims"][0]["id"]
    approve_candidate(driver, candidate_type="claim", node_id=cid, reviewer_id="expert_wang", comment="抽取准确，审核通过")
    print(f"✅ 已通过 claim {cid}")
if len(pending["claims"]) > 1:
    cid2 = pending["claims"][1]["id"]
    reject_candidate(driver, candidate_type="claim", node_id=cid2, reviewer_id="expert_wang", comment="证据不足，暂不采纳")
    print(f"❌ 已拒绝 claim {cid2}")

待审： {'strategies': 0, 'constructs': 0, 'claims': 0, 'results': 1}


In [6]:
# ============================================================
# Cell 6: 验证只有 approved 的 claim 进入技术简报
# ============================================================
from src.engineering_intelligence.technology_brief import generate_technology_brief
from src.engineering_intelligence.strategy_classifier import get_all_strategies

strategies = get_all_strategies(driver)
if strategies:
    brief = generate_technology_brief(driver, strategy_id=strategies[0]["id"], persist=False)
    print(f"技术简报包含的 Claim 数：{len(brief['technical_claims'])}")
    print(f"技术简报包含的 Result 数：{len(brief['technical_results'])}")
    print("（预期：只包含 approved 的，pending 的不会出现在这里）")

技术简报包含的 Claim 数：0
技术简报包含的 Result 数：0
（预期：只包含 approved 的，pending 的不会出现在这里）


In [7]:
# ============================================================
# Cell 7: 工程化技术图谱 / 策略对比 / White-space 分析
# ============================================================
from src.ci.competitor_profile import (
    map_engineering_landscape, compare_engineering_strategies, identify_landscape_white_space,
)

with driver.session() as s:
    pid_row = s.run("""
        MATCH (ec:EngineeredPhageConstruct)-[:TARGETS]->(p:Pathogen)
        RETURN p.pathogen_id AS pid, p.species AS species, count(ec) AS c
        ORDER BY c DESC LIMIT 1
    """).single()
if pid_row:
    land = map_engineering_landscape(driver, target_pathogen_id=pid_row["pid"])
    print(f"=== 技术图谱 ({pid_row['species']}) ===")
    for c in land["known_constructs"]:
        print(f"  {c['public_name']} [{c['construct_status']}] 策略={c['strategies']}")
    print(f"white_spaces: {len(land['white_spaces'])}")

with driver.session() as s:
    sids = [r["sid"] for r in s.run("MATCH (es:EngineeringStrategy) RETURN es.strategy_id AS sid ORDER BY es.strategy_type LIMIT 5")]
if sids:
    cmp = compare_engineering_strategies(driver, sids)
    print(f"\n=== 策略对比 === {len(cmp['strategies'])} 个策略")

ws = identify_landscape_white_space(driver)
print(f"\n=== White Space === 总组合数={ws['summary']['total_combinations']}, empty={ws['summary']['empty_count']}")

=== 技术图谱 (Pseudomonas aeruginosa) ===
  vB_Pae_PLD_010 [proposed] 策略=['payload_delivery']
  vB_Pae_BFD_007 [in_vitro_tested] 策略=['biofilm_disruption']
  vB_Pae_LYE_005 [in_vitro_tested] 策略=['lysis_enhancement']
  vB_Pae_HRE_002 [in_vitro_tested] 策略=['host_range_expansion']
white_spaces: 1

=== 策略对比 === 5 个策略

=== White Space === 总组合数=35, empty=25


In [8]:
# ============================================================
# Cell 8: PRD 工程对象缺口（DeliveryPayload + EngineeringModification）
# ============================================================
from src.engineering_intelligence.construct_service import (
    create_delivery_payload, list_payloads,
    create_engineering_modification, list_modifications,
)

with driver.session() as s:
    c = s.run("MATCH (ec:EngineeredPhageConstruct) RETURN ec.construct_id AS cid, ec.public_name AS n ORDER BY ec.public_name LIMIT 1").single()
if c:
    pl1 = create_delivery_payload(driver, payload_type="biofilm_enzyme",
        payload_description="Displays biofilm-degrading enzyme", construct_id=c["cid"])
    pl2 = create_delivery_payload(driver, payload_type="crispr_cas",
        payload_description="Encodes CRISPR-Cas3", construct_id=c["cid"])
    print(f"✅ Payloads: {pl1}, {pl2}")
    mod1 = create_engineering_modification(driver, modification_type="replacement",
        modification_summary="Replaced tail fiber gene with chimeric construct",
        construct_id=c["cid"], target_component="tail_fiber")
    mod2 = create_engineering_modification(driver, modification_type="insertion",
        modification_summary="Inserted lysin gene under late promoter",
        construct_id=c["cid"], target_component="lysis_module")
    print(f"✅ Modifications: {mod1}, {mod2}")

✅ Payloads: ENG:PAYLOAD:6BA013FC, ENG:PAYLOAD:EE2A9E35
✅ Modifications: ENG:MOD:D10C3343, ENG:MOD:1856259E


In [9]:
# ============================================================
# Cell 9: 技术简报 + 三个质量能力区块
# ============================================================
from src.engineering_intelligence.technology_brief import generate_technology_brief
from src.engineering_intelligence.strategy_classifier import get_all_strategies

strategies = get_all_strategies(driver)
target = strategies[0]
t_brief = generate_technology_brief(driver, strategy_id=target["id"],
    technology_question=f"{target['strategy_type']} 的公开证据成熟度如何？", persist=True)
t_bid = t_brief["brief_id"]
print(f"✅ Technology Brief: {t_bid}")

with driver.session() as s:
    row = s.run("""
        MATCH (b:IntelligenceProduct {brief_id: $bid})
        RETURN b.source_credibility AS src, b.conflict_warnings AS cf, b.related_patent_mappings AS pm
    """, bid=t_bid).single()

print("\n=== source_credibility ===")
print(json.dumps(json.loads(row["src"] or "{}"), ensure_ascii=False, indent=2))
print("\n=== conflict_warnings ===")
cf = json.loads(row["cf"] or "{}")
print(f"total_conflicts: {cf.get('total_conflicts', 0)}")
print("\n=== related_patent_mappings ===")
pm = json.loads(row["pm"] or "{}")
print(f"total_mappings: {pm.get('total_mappings', 0)}")

✅ Technology Brief: CI:BRIEF:311867CB

=== source_credibility ===
{
  "total_sources": 0,
  "by_tier": {
    "primary": 0,
    "secondary": 0,
    "tertiary": 0,
    "unknown": 0
  },
  "note": "来源可信度只反映渠道可靠性，不代表内容正确性。"
}

=== conflict_warnings ===
total_conflicts: 0

=== related_patent_mappings ===
total_mappings: 0
